In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

from skimage.measure import block_reduce
from helper_functions import write_text

In [ ]:
# Loading normal AGIPD detector mask and crop it around beam center
bg_mask = 'emc/make_detector/agipd_detector_mask.h5'
with h5py.File(bg_mask, 'r') as det:
    det_mask = det['mask'][:]
cy, cx = det_mask.shape[0] // 2, det_mask.shape[1] // 2
det_mask = det_mask[cy-cx:cy+cx]

d_mask_float = det_mask.astype(float)
d_mask_float[det_mask==False] = np.nan

det_mask_ds = d_mask_float
dsf = 4

In [ ]:
num_runs = 1
n_sim = 2000
def_rng = np.random.default_rng()
resamp = 4.0

for nr in range(num_runs):
    print(f'Resampling run {nr+1}/{num_runs}...')
    # Loading continuous scattering pattern and setting resampling factor
    run_file = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats/particle_intens.npy'
    particle_intens = np.load(run_file)[:]
    particle_intens *= resamp

    # Masking out detector panel gaps for continuous scattering patterns
    particle_intens_masked = particle_intens.copy()
    
    det_mask_ds_stack = np.broadcast_to(det_mask_ds, (n_sim,) + det_mask_ds.shape).astype(np.float64)
    particle_intens_masked[det_mask_ds_stack==False] = 0.0

    particle_intens_masked_ds = block_reduce(particle_intens_masked, block_size=dsf, func=np.nansum)

    # Poisson sampling continuous protein scattering pattern - both unmasked/masked
    poiss_samp = def_rng.poisson(lam=particle_intens)
    poiss_samp_masked = def_rng.poisson(lam=particle_intens_masked)

    # Saving resampled diffraction patterns
    file_name_particle_intens_masked = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats/particle_intens_masked_{int(resamp)}x_resampled.npy'
    file_name_poiss_samp = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats/poisson_prot_{int(resamp)}x_resampled.npy'
    file_name_poiss_samp_masked = f'sims_protein_water/run_{nr}_protein_in_water_100k_pats/poisson_prot_masked_{int(resamp)}x_resampled.npy'
    
    print("Saving resampled arrays...")
    #np.save(file_name_particle_intens_masked, arr=particle_intens_masked,)
    #np.save(file_name_poiss_samp, arr=poiss_samp,)
    #np.save(file_name_poiss_samp_masked, arr=poiss_samp_masked,)
    print("Finished saving resampling arrays...")

write_text('All resampling done...')